# Visualisasi Evaluasi Negasi Q02

Notebook ini membuat grafik khusus untuk evaluasi negasi berdasarkan file `negation_qq02_detail_ndcg_at_30.csv` dan `negation_qq02_summary_ndcg_at_30.csv`.

Grafik utama yang digunakan adalah **Score Decay & Relevance Mapping** agar konsisten dengan grafik pada evaluasi chunking. Grafik ini memperlihatkan penurunan skor kemiripan vektor pada setiap posisi ranking evaluasi, sekaligus memetakan nilai relevance dari ground truth.

Notebook ini juga menyediakan **Rank Shift Chart** sebagai grafik tambahan. Grafik tersebut memperlihatkan perubahan posisi ranking dari:

1. `rank_before_negation_filter`: ranking sebelum filter negasi diterapkan.
2. `rank_after_negation_filter`: ranking setelah filter negasi diterapkan.
3. `rank_after_eval_filter`: ranking setelah hasil dicocokkan dengan ground truth pool.

Catatan penting: pergeseran besar seperti rank 40 menjadi rank 4 adalah pergeseran menuju `rank_after_eval_filter`, yaitu setelah GT-pool filtering. Efek filter negasi murni dibaca dari `rank_before_negation_filter` ke `rank_after_negation_filter`.


In [ ]:
from pathlib import Path
import os
import textwrap

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 180,
    "axes.titleweight": "bold",
    "axes.labelsize": 10,
    "axes.titlesize": 12,
    "legend.fontsize": 8,
})

DETAIL_CSV = "negation_qq02_detail_ndcg_at_30.csv"
SUMMARY_CSV = "negation_qq02_summary_ndcg_at_30.csv"

# Bisa dijalankan dari root repo atau dari folder notebook.
candidate_dirs = [Path("."), Path("evaluation/negation_eval_results")]
DATA_DIR = next((p for p in candidate_dirs if (p / DETAIL_CSV).exists() and (p / SUMMARY_CSV).exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Tidak menemukan CSV evaluasi negasi. Jalankan notebook dari root repo atau folder negation_eval_results.")

FIG_DIR = DATA_DIR / "figures"
FIG_DIR.mkdir(exist_ok=True)

detail = pd.read_csv(DATA_DIR / DETAIL_CSV)
summary = pd.read_csv(DATA_DIR / SUMMARY_CSV)

numeric_cols = [
    "rank_after_eval_filter", "rank_after_negation_filter", "rank_before_negation_filter",
    "score", "relevance", "chunk_index", "total_chunks", "chunk_tokens",
    "gt_rta_id", "m", "ef_construct", "search_hnsw_ef", "raw_qdrant_limit",
    "gt_rows", "raw_hit_count", "unique_article_count", "after_negation_filter_count",
    "evaluated_rows", "matched_gt_rows_at_k", "dcg_at_30", "idcg_at_30",
    "ndcg_at_30", "elapsed_ms",
]
for df in [detail, summary]:
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

detail = detail.sort_values(["scenario", "rank_after_eval_filter"]).reset_index(drop=True)
detail["relevance_label"] = detail["relevance"].fillna(0).astype(int).astype(str)
detail["doc_label"] = (
    detail["agreement"].astype(str)
    + " | " + detail["article_number"].astype(str)
    + " | " + detail["article_name"].astype(str)
)
detail["doc_short"] = detail["doc_label"].map(lambda s: textwrap.shorten(s, width=72, placeholder="..."))
detail["negation_rank_shift"] = detail["rank_before_negation_filter"] - detail["rank_after_negation_filter"]
detail["gt_pool_rank_shift"] = detail["rank_after_negation_filter"] - detail["rank_after_eval_filter"]
SCENARIO_LABELS = {
    "negation_on_finetuned": "negation on",
    "negation_off": "negation off",
}

print(f"Folder data: {DATA_DIR.resolve()}")
print(f"Detail rows: {len(detail)}")
print(f"Figure output: {FIG_DIR.resolve()}")


## Ringkasan Evaluasi

Tabel ini menjadi konteks sebelum membaca grafik. Nilai NDCG@30 tetap dibaca sebagai GT-filtered NDCG@30 sesuai metode evaluasi yang digunakan.


In [ ]:
summary[[
    "scenario", "final_query", "excluded_terms", "gt_rows",
    "raw_hit_count", "unique_article_count", "after_negation_filter_count",
    "evaluated_rows", "matched_gt_rows_at_k", "ndcg_at_30", "elapsed_ms"
]]


## Grafik 1 - Score Decay & Relevance Mapping

Grafik ini dibuat dengan format yang sama seperti evaluasi chunking. Sumbu X menunjukkan `rank_after_eval_filter`, sumbu Y menunjukkan skor kemiripan vektor (`score`), garis menunjukkan tren penurunan skor per skenario, dan warna marker menunjukkan nilai relevance dari ground truth.

Karena data detail ini menggunakan pendekatan GT-filtered, grafik ini menjelaskan kualitas ranking pada artikel yang masuk ground truth pool.


In [ ]:
score_decay = detail.copy().sort_values(["scenario", "rank_after_eval_filter"])

scenario_colors = {
    "negation_on_finetuned": "#2563eb",
    "negation_off": "#16a34a",
}
scenario_markers = {
    "negation_on_finetuned": "o",
    "negation_off": "s",
}
relevance_colors = {
    "0": "#dc2626",
    "1": "#f97316",
    "2": "#eab308",
    "3": "#15803d",
}
relevance_sizes = {
    "0": 50,
    "1": 70,
    "2": 95,
    "3": 125,
}

fig, ax = plt.subplots(figsize=(10, 5.8))

for scenario, scenario_df in score_decay.groupby("scenario"):
    ax.plot(
        scenario_df["rank_after_eval_filter"],
        scenario_df["score"],
        color=scenario_colors.get(scenario, "#64748b"),
        linewidth=2,
        alpha=0.72,
        label=SCENARIO_LABELS.get(scenario, scenario.replace("_", " ")),
        zorder=1,
    )

for scenario, scenario_df in score_decay.groupby("scenario"):
    marker = scenario_markers.get(scenario, "o")
    for rel_label, rel_df in scenario_df.groupby("relevance_label"):
        ax.scatter(
            rel_df["rank_after_eval_filter"],
            rel_df["score"],
            s=relevance_sizes.get(rel_label, 70),
            marker=marker,
            color=relevance_colors.get(rel_label, "#64748b"),
            edgecolor="white",
            linewidth=0.9,
            alpha=0.94,
            zorder=3,
        )

for _, row in score_decay.iterrows():
    ax.annotate(
        str(int(row["relevance"])),
        xy=(row["rank_after_eval_filter"], row["score"]),
        xytext=(0, 7),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=7,
        color="#111827",
    )

ax.set_xlabel("Rank setelah filter evaluasi")
ax.set_ylabel("Vector similarity score")
ax.set_title("Score Decay & Ground Truth Relevance Mapping")
ax.set_xticks(sorted(score_decay["rank_after_eval_filter"].dropna().astype(int).unique()))
ax.grid(True, axis="y", alpha=0.35)

scenario_handles = [
    plt.Line2D([0], [0], color=color, linewidth=2.2, label=f"Score decay - {SCENARIO_LABELS.get(scenario, scenario.replace('_', ' '))}")
    for scenario, color in scenario_colors.items() if scenario in score_decay["scenario"].unique()
]
relevance_handles = [
    plt.Line2D([0], [0], marker="o", color="none", label=f"Relevance {label}",
               markerfacecolor=color, markeredgecolor="white", markersize=8)
    for label, color in relevance_colors.items() if label in score_decay["relevance_label"].unique()
]

ax.legend(handles=scenario_handles + relevance_handles, bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0, frameon=True)

plt.tight_layout(rect=[0, 0, 0.82, 1])

score_decay_path = FIG_DIR / "negation_q02_score_decay_relevance_mapping.png"
fig.savefig(score_decay_path, bbox_inches="tight")
plt.show()

score_decay_path


In [ ]:
score_decay_summary = (
    score_decay.groupby("scenario", as_index=False)
    .agg(
        evaluated_rows=("rank_after_eval_filter", "count"),
        first_score=("score", "first"),
        last_score=("score", "last"),
        mean_score=("score", "mean"),
        relevance_sum=("relevance", "sum"),
        mean_relevance=("relevance", "mean"),
    )
)
score_decay_summary


## Grafik Tambahan - Rank Shift Chart

Grafik ini memperlihatkan perubahan ranking setiap artikel pada tiga tahap. Sumbu Y dibalik karena ranking yang lebih kecil berarti posisi yang lebih baik. Warna garis menunjukkan nilai relevance dari ground truth.

- Pergeseran dari **Before Negation** ke **After Negation** menunjukkan efek filter negasi.
- Pergeseran dari **After Negation** ke **GT-filtered Eval Rank** menunjukkan kompresi ranking setelah hanya artikel dalam ground truth pool yang dipakai dalam evaluasi.


In [ ]:
stage_cols = [
    ("Before negation filter", "rank_before_negation_filter"),
    ("After negation filter", "rank_after_negation_filter"),
    ("GT-filtered eval rank", "rank_after_eval_filter"),
]
stage_x = {name: idx for idx, (name, _) in enumerate(stage_cols)}
relevance_palette = {
    "0": "#dc2626",
    "1": "#f97316",
    "2": "#eab308",
    "3": "#15803d",
}

scenarios = summary["scenario"].tolist()
fig, axes = plt.subplots(1, len(scenarios), figsize=(6.2 * len(scenarios), 7), sharey=True)
if len(scenarios) == 1:
    axes = [axes]

max_rank = int(detail[[col for _, col in stage_cols]].max().max())

for ax, scenario in zip(axes, scenarios):
    scenario_df = detail[detail["scenario"] == scenario].copy()
    scenario_df["_annotate"] = False
    annotate_idx = (
        scenario_df.sort_values(["gt_pool_rank_shift", "negation_rank_shift"], ascending=[False, False])
        .head(4)
        .index
    )
    scenario_df.loc[annotate_idx, "_annotate"] = True
    for _, row in scenario_df.iterrows():
        xs = [stage_x[name] for name, _ in stage_cols]
        ys = [row[col] for _, col in stage_cols]
        color = relevance_palette.get(row["relevance_label"], "#64748b")
        ax.plot(xs, ys, color=color, linewidth=1.5, alpha=0.55)
        ax.scatter(xs, ys, color=color, edgecolor="white", linewidth=0.7, s=52, alpha=0.95, zorder=3)

        # Anotasi hanya untuk beberapa perubahan terbesar agar grafik tetap mudah dibaca.
        if row["_annotate"]:
            label = f"eval #{int(row['rank_after_eval_filter'])}\nrel {int(row['relevance'])}"
            ax.annotate(
                label,
                xy=(stage_x["GT-filtered eval rank"], row["rank_after_eval_filter"]),
                xytext=(7, 0),
                textcoords="offset points",
                va="center",
                fontsize=7,
                color="#111827",
            )

    ax.set_xticks(list(stage_x.values()))
    ax.set_xticklabels(list(stage_x.keys()), rotation=18, ha="right")
    ax.set_ylim(max_rank + 5, 0)
    ax.set_title(scenario.replace("_", " ").title())
    ax.set_xlabel("Tahap ranking")
    ax.grid(True, axis="y", alpha=0.28)

axes[0].set_ylabel("Ranking (lebih kecil lebih baik)")

handles = [
    plt.Line2D([0], [0], marker="o", color=color, label=f"Relevance {label}",
               markerfacecolor=color, markeredgecolor="white", linewidth=1.8)
    for label, color in relevance_palette.items()
]
fig.legend(handles=handles, title="Ground truth relevance", loc="upper center", ncol=4, bbox_to_anchor=(0.5, 1.02))
fig.suptitle("Rank Shift Chart Evaluasi Negasi Q02", y=1.09, fontsize=13, fontweight="bold")
plt.tight_layout()

rank_shift_path = FIG_DIR / "negation_q02_rank_shift_chart.png"
fig.savefig(rank_shift_path, bbox_inches="tight")
plt.show()

rank_shift_path


## Grafik 2 - Dual Line Rank Comparison

Grafik ini adalah versi yang lebih sederhana. Sumbu X adalah ranking evaluasi setelah GT filtering, sedangkan sumbu Y membandingkan ranking sebelum dan sesudah filter negasi. Jika garis setelah filter berada lebih tinggi pada grafik, berarti ranking numeriknya lebih kecil atau lebih baik.


In [ ]:
fig, axes = plt.subplots(1, len(scenarios), figsize=(6.2 * len(scenarios), 5.2), sharey=True)
if len(scenarios) == 1:
    axes = [axes]

for ax, scenario in zip(axes, scenarios):
    scenario_df = detail[detail["scenario"] == scenario].sort_values("rank_after_eval_filter")
    ax.plot(
        scenario_df["rank_after_eval_filter"],
        scenario_df["rank_before_negation_filter"],
        marker="o",
        linewidth=2,
        label="Before negation filter",
        color="#2563eb",
    )
    ax.plot(
        scenario_df["rank_after_eval_filter"],
        scenario_df["rank_after_negation_filter"],
        marker="s",
        linewidth=2,
        label="After negation filter",
        color="#dc2626",
    )
    ax.scatter(
        scenario_df["rank_after_eval_filter"],
        scenario_df["rank_after_eval_filter"],
        s=35,
        marker="x",
        color="#111827",
        label="GT-filtered eval rank",
        zorder=4,
    )
    ax.set_title(scenario.replace("_", " ").title())
    ax.set_xlabel("Rank setelah GT-filtered evaluation")
    ax.set_xticks(scenario_df["rank_after_eval_filter"].astype(int).tolist())
    ax.set_ylim(max_rank + 5, 0)
    ax.grid(True, axis="y", alpha=0.3)

axes[0].set_ylabel("Ranking di hasil retrieval (lebih kecil lebih baik)")
axes[-1].legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=True)
fig.suptitle("Perbandingan Ranking Sebelum dan Sesudah Filter Negasi", y=1.04, fontsize=13, fontweight="bold")
plt.tight_layout()

dual_line_path = FIG_DIR / "negation_q02_dual_line_rank_comparison.png"
fig.savefig(dual_line_path, bbox_inches="tight")
plt.show()

dual_line_path


## Tabel Pergeseran Ranking

Tabel ini membantu membaca grafik. `negation_rank_shift` adalah perubahan ranking yang benar-benar disebabkan oleh filter negasi. `gt_pool_rank_shift` adalah perubahan ranking akibat artikel di luar ground truth pool tidak dimasukkan dalam urutan evaluasi.


In [ ]:
shift_summary = (
    detail.groupby("scenario", as_index=False)
    .agg(
        evaluated_rows=("rank_after_eval_filter", "count"),
        mean_relevance=("relevance", "mean"),
        relevance_sum=("relevance", "sum"),
        mean_negation_rank_shift=("negation_rank_shift", "mean"),
        max_negation_rank_shift=("negation_rank_shift", "max"),
        mean_gt_pool_rank_shift=("gt_pool_rank_shift", "mean"),
        max_gt_pool_rank_shift=("gt_pool_rank_shift", "max"),
    )
)
shift_summary


In [ ]:
detail.sort_values(["scenario", "gt_pool_rank_shift", "negation_rank_shift"], ascending=[True, False, False])[[
    "scenario", "rank_before_negation_filter", "rank_after_negation_filter",
    "rank_after_eval_filter", "negation_rank_shift", "gt_pool_rank_shift",
    "relevance", "score", "doc_short"
]]


## Catatan Interpretasi

Grafik rank shift ini paling tepat dipakai untuk menjelaskan perubahan posisi ranking akibat filter negasi dan akibat mekanisme evaluasi GT-filtered. Untuk klaim efek filter negasi, gunakan perubahan dari `rank_before_negation_filter` ke `rank_after_negation_filter`. Untuk menjelaskan mengapa dokumen yang ranking awalnya jauh tetap masuk evaluasi, gunakan `rank_after_eval_filter`, karena ranking tersebut dihitung setelah dokumen di luar ground truth pool diabaikan dari urutan evaluasi.
